<a href="https://colab.research.google.com/github/marinhomar/blank-app/blob/main/parte1_extracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import unicodedata

In [3]:
BASE = "https://raw.githubusercontent.com/marinhomar/cardioia-fase2/main/src/parte1/"

frases = pd.read_csv(BASE + "frases.txt", header=None, names=["frase"], sep="\t")
mapa = pd.read_csv(BASE + "mapa_conhecimento.csv")
sinonimos = pd.read_csv(BASE + "sinonimos.csv")

frases


,frase
0,Desde hoje cedo sinto uma dor no peito forte c...
1,Há duas horas estou com um aperto no tórax e d...
2,Desde a madrugada sinto dor no queixo e náusea...
3,"Há três semanas sinto dor ao subir escada, mas..."
4,Há um mês tenho dor ao fazer esforço na academ...
5,Faz dez dias que sinto falta de ar e dificulda...
6,Há uma semana sinto cansaço constante mesmo de...
7,Há quinze dias estou com as pernas inchadas e ...
8,Desde a semana passada acordo sem ar durante a...
9,Há um mês vejo que ganho peso rápido e barriga...


In [4]:
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

In [5]:
sinonimos["variante"] = sinonimos["variante"].apply(normalizar)
sinonimos["termo_padrao"] = sinonimos["termo_padrao"].apply(normalizar)
sinonimos = sinonimos.sort_values("variante", key=lambda s: s.str.len(), ascending=False)

def traduzir(texto):
    for variante, padrao in zip(sinonimos["variante"], sinonimos["termo_padrao"]):
        texto = texto.replace(variante, padrao)
    return texto

In [6]:
GRAVIDADE = ["Infarto", "Angina", "Insuficiência Cardíaca"]

def diagnosticar(frase):
    texto = traduzir(normalizar(frase))
    pontos = {}
    encontrados = []
    for _, linha in mapa.iterrows():
        for sintoma in (linha["Sintoma 1"], linha["Sintoma 2"]):
            if normalizar(sintoma) in texto:
                encontrados.append(sintoma)
                doenca = linha["Doença Associada"]
                pontos[doenca] = pontos.get(doenca, 0) + 1
    if not pontos:
        return encontrados, "Sem diagnóstico"
    maior = max(pontos.values())
    empatadas = [d for d in pontos if pontos[d] == maior]
    if len(empatadas) == 1:
        return encontrados, empatadas[0]
    prioridade = min(empatadas, key=GRAVIDADE.index)
    return encontrados, f"Inconclusivo ({' / '.join(empatadas)}) → priorizar: {prioridade}"

In [7]:
resultado = frases.copy()
resultado[["sintomas_encontrados", "diagnostico"]] = resultado["frase"].apply(
    lambda f: pd.Series(diagnosticar(f))
)
pd.set_option("display.max_colwidth", 120)
resultado

,frase,sintomas_encontrados,diagnostico
0,Desde hoje cedo sinto uma dor no peito forte com suor frio e não consegui sair da cama para trabalhar.,"[dor no peito, suor frio]",Infarto
1,"Há duas horas estou com um aperto no tórax e dor no braço esquerdo, tive que parar de dirigir no meio do caminho.","[aperto no tórax, dor no braço esquerdo]",Infarto
2,"Desde a madrugada sinto dor no queixo e náusea, e não me sinto bem.","[dor no queixo, náusea]",Infarto
3,"Há três semanas sinto dor ao subir escada, mas passa quando descanso; agora evito ir ao segundo andar do escritório.","[dor ao subir escada, passa quando descanso]",Angina
4,"Há um mês tenho dor ao fazer esforço na academia e ela melhora com repouso, por isso parei de treinar.","[dor ao fazer esforço, melhora com repouso]",Angina
5,"Faz dez dias que sinto falta de ar e dificuldade para respirar quando caminho rápido, e deixei de ir a pé para o tra...","[falta de ar, dificuldade para respirar]",Angina
6,Há uma semana sinto cansaço constante mesmo depois de descansar e não consigo terminar meu dia bem.,[cansaço constante],Insuficiência Cardíaca
7,"Há quinze dias estou com as pernas inchadas e os tornozelos inchados no fim do dia, e meus sapatos não servem mais.","[pernas inchadas, tornozelos inchados]",Insuficiência Cardíaca
8,"Desde a semana passada acordo sem ar durante a noite e preciso de três travesseiros, então durmo mal e chego exausto...",[acordo sem ar],Insuficiência Cardíaca
9,"Há um mês vejo que ganho peso rápido e barriga inchada, além de muita fadiga, e já não consigo brincar com meu neto.","[fadiga, ganho peso rápido, barriga inchada]",Insuficiência Cardíaca


In [8]:
diagnosticar("Desde ontem tô com uma agonia no peito e suando frio, não fui trabalhar.")

(['dor no peito', 'suor frio'], 'Infarto')